In [1]:
# 本小节是将01函数调用和02语言表达式的内容进行结合
import os, openai

api_key = os.environ.get("DEEPSEEK_API_KEY")
model_name = "deepseek-v4-flash"
url = "https://api.deepseek.com/"

In [2]:
# 定义一个Pydantic类而不使用init方法
from pydantic import BaseModel, Field
from typing import List

In [3]:
# 创建一个普通类
class Users:
    def __init__(self, name: str, age: int, email: str):
        self.name = name
        self.age = age
        self.email = email

In [4]:
foo = Users(name="Joe", age=32, email="joe@gmail.com")

In [5]:
foo.name

'Joe'

In [6]:
foo = Users(name="Joe" ,age="bar" ,email="joe@gmail.com")

In [7]:
foo.age

'bar'

In [8]:
# 使用Pydantic
class pUser(BaseModel):
    name: str
    age: int
    email: str
    # 指定了类型

In [9]:
foo_p = pUser(name="Joe" ,age=32 ,email="joe@gmail.com")

In [10]:
foo_p.name

'Joe'

In [11]:
# foo_p = pUser(name="Joe" ,age="bar" ,email="joe@gmail.com")

In [12]:
foo_p.age

32

In [13]:
# Pydantic同时可以嵌套这些数据结构
class Class(BaseModel):
    students: List[pUser] # 创建一个pUser类型的列表

In [14]:
obj = Class(
    students=[pUser(name="Jane", age=32, email="jane@gmail.com")] #创建对象传入学生列表
)

In [15]:
obj

Class(students=[pUser(name='Jane', age=32, email='jane@gmail.com')])

In [16]:
# 使用Pydantic去创建OpenAI定义
# 转换为json

class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str = Field(description="airport code to get weather for")

In [42]:
from langchain_classic.utils.openai_functions import convert_pydantic_to_openai_tool

In [43]:
weather_fuction = convert_pydantic_to_openai_tool(WeatherSearch)
# 此处传递的是类名，而没有使用具体的对象初始化

In [19]:
weather_fuction

{'name': 'WeatherSearch',
 'description': 'Call this with an airport code to get the weather at that airport',
 'parameters': {'properties': {'airport_code': {'description': 'airport code to get weather for',
    'type': 'string'}},
  'required': ['airport_code'],
  'type': 'object'}}

In [20]:
class WeatherSearch1(BaseModel):
    # 此处的文档在旧版本当中不可缺少，而在新的版本的openai当中函数的Description可以为空
    airport_code: str = Field(description="airport code to get weather for")

In [21]:
convert_pydantic_to_openai_tool(WeatherSearch1)

{'name': 'WeatherSearch1',
 'description': '',
 'parameters': {'properties': {'airport_code': {'description': 'airport code to get weather for',
    'type': 'string'}},
  'required': ['airport_code'],
  'type': 'object'}}

In [22]:
class WeatherSearch2(BaseModel):
    # 此处的文档在旧版本当中不可缺少，而在新的版本的openai当中函数的Description可以为空
    airport_code: str

In [23]:
convert_pydantic_to_openai_tool(WeatherSearch2)

{'name': 'WeatherSearch2',
 'description': '',
 'parameters': {'properties': {'airport_code': {'type': 'string'}},
  'required': ['airport_code'],
  'type': 'object'}}

In [24]:
from langchain_openai import ChatOpenAI

In [25]:
model = ChatOpenAI(
    base_url=url,
    api_key=api_key,
    model=model_name,
)

In [26]:
# 内联传递工具调用
response = model.invoke(
    "What is the weather like in SF today",
    tools=[
        {
            "type":"function",
            "function": weather_fuction
        }
    ],
)
# 在新版本的langchain当中，function写法需要修改为bind_tools写法，为模型绑定函数方法

In [27]:
response.tool_calls # 无需每次传递函数关键字

[{'name': 'WeatherSearch',
  'args': {'airport_code': 'SFO'},
  'id': 'call_00_UM4qIiHXZ5Tf8TY5wkjh1399',
  'type': 'tool_call'}]

In [28]:
response

AIMessage(content="I'll look up the weather for San Francisco using its airport code.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 382, 'total_tokens': 476, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 34, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 126}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '98b092ed-f627-437c-a052-97492e10174f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0806e-a43d-7110-9f2c-76a0226cfbaa-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_00_UM4qIiHXZ5Tf8TY5wkjh1399', 'type': 'to

In [29]:
model_with_function = model.bind_tools([WeatherSearch]) # 区别bind和bind_tools

In [30]:
response = model_with_function.invoke("What's the weather like in SF?")

In [31]:
response.tool_calls

[{'name': 'WeatherSearch',
  'args': {'airport_code': 'SFO'},
  'id': 'call_00_U4xVCL7ETO9hOc62xAij7354',
  'type': 'tool_call'}]

In [32]:
# 强制使用模型调用函数
model_with_forced_function = model.bind_tools([weather_fuction], tool_choice="WeatherSearch")

In [33]:
# model_with_forced_function.invoke("what is the weather in sf?")

In [34]:
# 在chain中使用
from langchain_classic.prompts import ChatPromptTemplate

In [35]:
prompt = ChatPromptTemplate.from_messages([
    # 创建一个简单的系统消息
    ("system", "You are a helpful assistant"),
    ("user", "{input}")
])

In [36]:
# 创建一条链
chain = prompt | model_with_function

In [37]:
chain.invoke({"input": "what is the weather in sf?"})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 387, 'total_tokens': 473, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 39, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 384, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 3}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '690f8591-bada-46fe-bdf9-b8f321958818', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0806e-ae8e-7e82-ae61-7f8e3752a258-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'SFO'}, 'id': 'call_00_jtzui1RZMycgQNwg5OEh3529', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3

In [38]:
# 传递函数列表，让语言模型根据问题上下文选择使用哪个函数
class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""
    artist_name: str = Field(description="name of artist to look up")
    n: int = Field(description="number of results") # 查找结果数量值

In [44]:
functions = [
    convert_pydantic_to_openai_tool(WeatherSearch),
    convert_pydantic_to_openai_tool(ArtistSearch),
]

In [45]:
model_with_function = model.bind(tools=functions)

In [46]:
response = model_with_function.invoke("what is the weather in sf?")

In [47]:
response.tool_calls

[{'name': 'WeatherSearch',
  'args': {'airport_code': 'SFO'},
  'id': 'call_00_WcqwnXOartViV8GtmRnV8532',
  'type': 'tool_call'}]

In [48]:
response = model_with_function.invoke("what are three songs by taylor swift?")

In [49]:
response.tool_calls

[{'name': 'ArtistSearch',
  'args': {'artist_name': 'Taylor Swift', 'n': 3},
  'id': 'call_00_d71fdGlGY22tPhBHi8p37411',
  'type': 'tool_call'}]

In [50]:
# 写法2
model_with_function = model.bind_tools(functions)


In [51]:
response = model_with_function.invoke("what are three songs by One Republic")

In [55]:
response.tool_calls[0]["args"]

{'artist_name': 'OneRepublic', 'n': 3}

In [53]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 467, 'total_tokens': 552, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 23, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 384, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 83}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '5ada7d06-748d-4717-bcb0-06509d814ee6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a08071-a2e2-7ce3-a871-1e74b4f81d9e-0', tool_calls=[{'name': 'ArtistSearch', 'args': {'artist_name': 'OneRepublic', 'n': 3}, 'id': 'call_00_lTWJb2eNr61smNwLAsod2100', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i

In [57]:
model_with_function.invoke("hi!").content

"Hi there! 😊 How can I help you today?\n\nI can look up weather conditions at various airports, or search for songs by a particular artist. Let me know what you'd like to explore!"